# DeepLearning.AI

## LangChain Expression Language (LCEL)

<img src="./pics/functions-tools-agents-langchain.png" width=800> 

https://learn.deeplearning.ai/courses/functions-tools-agents-langchain/information

In [1]:
pwd

'c:\\Users\\crodr\\aiDeepLearning\\functions-tools-agents-langchain'

In [2]:
!python --version


Python 3.13.13


In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")

# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


In [4]:
import langchain

print(langchain.__version__)

1.2.18


New Routes for packages in LangChain v.1.2+

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


### Simple Chain

In [ ]:
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")

In [7]:
model = ChatOpenAI(model="gpt-4o-mini")
output_parser = StrOutputParser()

In [8]:
chain = prompt | model | output_parser

In [9]:
chain.invoke({"topic": "bears"})

'Why do bears have hairy coats? \n\nBecause they look silly in sweaters!'

### More complex chain
And RunnableParallel to supply user-provided inputs to the prompt.

**Note:**

To install safely: in the VS code - terminal run:

`uv pip install langchain-community docarray --python .venv/Scripts/python.exe`

In [10]:
# from langchain.embeddings import OpenAIEmbeddings
# from langchain.vectorstors import DocArrayInMemorySearch
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch

In [ ]:
vectorstore = DocArrayInMemorySearch.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever()

c:\Users\crodr\aiDeepLearning\functions-tools-agents-langchain\.venv\Lib\site-packages\docarray\helper.py:255: SyntaxWarning: invalid escape sequence '\*'
  e.g. '\*.py', '[\*.zip, \*.gz]'


In [14]:
# retriever._get_relevant_documents("Where did harrison work?")
retriever.invoke("Where did harrison work?")

[Document(metadata={}, page_content='harrison worked at kensho'),
 Document(metadata={}, page_content='bears like to eat honey')]

In [15]:
retriever.invoke("What do bears like to eat?")


[Document(metadata={}, page_content='bears like to eat honey'),
 Document(metadata={}, page_content='harrison worked at kensho')]

In [16]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [18]:
# from langchain.schema.runnable import RunnableMap
from langchain_core.runnables import RunnableParallel

In [ ]:
# chain = RunnableMap({
#     "context": lambda x: retriever.get_relevant_documents(x["question"]),
#     "question": lambda x: x["question"]
# }) | prompt | model | output_parser

In [ ]:
chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
    }
    | prompt
    | model
    | output_parser
)

In [22]:
chain.invoke({"question": "where did harrison work?"})

'Harrison worked at Kensho.'

In [ ]:
inputs = {
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"],
} | prompt

In [32]:
inputs.invoke({"question": "where did harrison work?"})


ChatPromptValue(messages=[HumanMessage(content="Answer the question based only on the following context:\n[Document(metadata={}, page_content='harrison worked at kensho'), Document(metadata={}, page_content='bears like to eat honey')]\n\nQuestion: where did harrison work?\n", additional_kwargs={}, response_metadata={})])

### Bind
and OpenAI Functions

In [6]:
functions = [
    {
        "name": "weather_search",
        "description": "Search for weather given an airport code",
        "parameters": {
            "type": "object",
            "properties": {
                "airport_code": {
                    "type": "string",
                    "description": "The airport code to get the weather for",
                },
            },
            "required": ["airport_code"],
        },
    }
]

In [8]:
prompt = ChatPromptTemplate.from_messages([("human", "{input}")])
model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind(functions=functions)


In [9]:
runnable = prompt | model

In [10]:
runnable.invoke({"input": "what is the weather in sf"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'weather_search'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 62, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3b264ad91d', 'id': 'chatcmpl-DddY17pXriWuXCPRfQFcdEdc2u8Lu', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e0d46-7931-7642-a44e-04a700d39b1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 16, 'total_tokens': 78, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [11]:
functions = [
    {
        "name": "weather_search",
        "description": "Search for weather given an airport code",
        "parameters": {
            "type": "object",
            "properties": {
                "airport_code": {
                    "type": "string",
                    "description": "The airport code to get the weather for",
                },
            },
            "required": ["airport_code"],
        },
    },
    {
        "name": "sports_search",
        "description": "Search for news of recent sport events",
        "parameters": {
            "type": "object",
            "properties": {
                "team_name": {
                    "type": "string",
                    "description": "The sports team to search for",
                },
            },
            "required": ["team_name"],
        },
    },
]

In [13]:
model = model.bind(functions=functions)

In [14]:
runnable = prompt | model

In [15]:
runnable.invoke({"input": "how did the patriots do yesterday?"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"team_name":"New England Patriots"}', 'name': 'sports_search'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 95, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6b1e1d7ece', 'id': 'chatcmpl-DddjsCjiS8NDM1OIn6JS2fPXn48St', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e0d51-b580-7163-abec-95d2947fb181-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 18, 'total_tokens': 113, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Fallbacks

In [16]:
# from langchain.llms import OpenAI
from langchain_openai import OpenAI
import json


In [ ]:
simple_model = OpenAI(temperature=0, max_tokens=1000, model="gpt-3.5-turbo-instruct")
simple_chain = simple_model | json.loads

In [18]:
challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line"

In [19]:
simple_model.invoke(challenge)

'\n\n{\n    "title": "Autumn Leaves",\n    "author": "Emily Dickinson",\n    "first_line": "The leaves are falling, one by one"\n}\n\n{\n    "title": "The Ocean\'s Song",\n    "author": "Pablo Neruda",\n    "first_line": "I hear the ocean\'s song, a symphony of waves"\n}\n\n{\n    "title": "A Winter\'s Night",\n    "author": "Robert Frost",\n    "first_line": "The snow falls softly, covering the ground"\n}'

In [20]:
model = ChatOpenAI(temperature=0)
chain = model | StrOutputParser() | json.loads

In [21]:
chain.invoke(challenge)

{'poem1': {'title': 'The Rose',
  'author': 'Emily Dickinson',
  'firstLine': 'A rose by any other name would smell as sweet'},
 'poem2': {'title': 'The Road Not Taken',
  'author': 'Robert Frost',
  'firstLine': 'Two roads diverged in a yellow wood'},
 'poem3': {'title': 'Hope is the Thing with Feathers',
  'author': 'Emily Dickinson',
  'firstLine': 'Hope is the thing with feathers that perches in the soul'}}

In [22]:
final_chain = simple_chain.with_fallbacks([chain])

In [23]:
final_chain.invoke(challenge)

{'poem1': {'title': 'The Rose',
  'author': 'Emily Dickinson',
  'firstLine': 'A rose by any other name would smell as sweet'},
 'poem2': {'title': 'The Road Not Taken',
  'author': 'Robert Frost',
  'firstLine': 'Two roads diverged in a yellow wood'},
 'poem3': {'title': 'Hope is the Thing with Feathers',
  'author': 'Emily Dickinson',
  'firstLine': 'Hope is the thing with feathers that perches in the soul'}}

### Interface

In [ ]:
prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")

model = ChatOpenAI()
output_parser = StrOutputParser()

chain = prompt | model | output_parser

In [27]:
chain.invoke({"topic": "bears"})

'Why did the bear bring a flashlight to the party? \n\nBecause he heard it was going to be a "beary" good time!'

In [29]:
chain.batch([{"topic": "bears"}, {"topic": "frogs"}])


["Why don't bears like fast food? Because they can't catch it!",
 'Why are frogs so happy? Because they eat whatever bugs them!']

In [30]:
for t in chain.stream({"topic": "bears"}):
    print(t)


Why
 did
 the
 bear
 bring
 a
 flashlight
 to
 the
 party
?

Because
 he
 heard
 it
 was
 going
 to
 be
 a
 "
bear
"
y
 dark
 night
!





In [32]:
response = await chain.ainvoke({"topic": "bears"})
response

"Why did the bear wear a fur coat to the party?\nBecause he didn't want to be under-dressed!"